In [6]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import joblib, os


# 데이터 로드
train  = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')
test   = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')
layout = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/layout_info.csv')

# layout merge만 (building_age_years 제외)
layout_clean = layout[['layout_id', 'layout_type', 'aisle_width_avg',
                        'intersection_count', 'one_way_ratio', 'pack_station_count',
                        'charger_count', 'layout_compactness', 'zone_dispersion',
                        'robot_total', 'floor_area_sqm', 'ceiling_height_m',
                        'fire_sprinkler_count', 'emergency_exit_count']]

train = train.merge(layout_clean, on='layout_id', how='left')
test  = test.merge(layout_clean, on='layout_id', how='left')

# layout_type 인코딩
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train['layout_type'] = le.fit_transform(train['layout_type'].fillna('unknown'))
test['layout_type']  = le.transform(test['layout_type'].fillna('unknown'))

TARGET  = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f'feature 수: {len(feature_cols)}')
print(f'train: {train.shape}, test: {test.shape}')

feature 수: 103
train: (250000, 107), test: (50000, 106)


In [7]:
# 1단계: best iteration 확인 (split으로)
from sklearn.model_selection import train_test_split

X = train[feature_cols]
y = train[TARGET]

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model_step1 = lgb.LGBMRegressor(
    n_estimators     = 100000,
    learning_rate    = 0.05,
    num_leaves       = 63,
    max_depth        = 7,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 0.1,
    objective        = 'mae',
    metric           = 'mae',
    random_state     = 42,
    n_jobs           = -1,
    verbose          = -1,
)

model_step1.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(300), lgb.log_evaluation(5000)]
)

best_iter = model_step1.best_iteration_
print(f'Best iteration: {best_iter}')
mae = np.mean(np.abs(y_val - model_step1.predict(X_val)))
print(f'Validation MAE: {mae:.4f}')

Training until validation scores don't improve for 300 rounds
[5000]	valid_0's l1: 7.28093
[10000]	valid_0's l1: 7.20209
[15000]	valid_0's l1: 7.17694
[20000]	valid_0's l1: 7.16295
[25000]	valid_0's l1: 7.14976
[30000]	valid_0's l1: 7.14117
[35000]	valid_0's l1: 7.13408
[40000]	valid_0's l1: 7.12543
[45000]	valid_0's l1: 7.12267
[50000]	valid_0's l1: 7.11472
[55000]	valid_0's l1: 7.10695
[60000]	valid_0's l1: 7.10117
[65000]	valid_0's l1: 7.09198
[70000]	valid_0's l1: 7.08683
Early stopping, best iteration is:
[72924]	valid_0's l1: 7.08434
Best iteration: 72924
Validation MAE: 7.0843


In [8]:
# 2단계: best iteration 고정 + KFold 전체 데이터 학습
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds  = np.zeros(len(train))
test_preds = np.zeros(len(test))

kfold_params = {
    'n_estimators'     : 72924,  # best iteration 고정
    'learning_rate'    : 0.05,
    'num_leaves'       : 63,
    'max_depth'        : 7,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 0.1,
    'objective'        : 'mae',
    'metric'           : 'mae',
    'random_state'     : 42,
    'n_jobs'           : -1,
    'verbose'          : -1,
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f'── Fold {fold+1} ──')
    X_tr,  y_tr  = X.iloc[tr_idx],  y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**kfold_params)
    model.fit(X_tr, y_tr)  # early stopping 없이 고정 iteration

    joblib.dump(model, f'C:/Users/user/dacon/Smart_Logistics/models/lgbm_simple_fold{fold+1}.pkl')

    oof_preds[val_idx] = model.predict(X_val)
    test_preds        += model.predict(test[feature_cols]) / 5

    fold_mae = np.mean(np.abs(y_val - model.predict(X_val)))
    print(f'Fold {fold+1} MAE: {fold_mae:.4f}')

oof_mae = np.mean(np.abs(y - oof_preds))
print(f'\nOOF MAE: {oof_mae:.4f}')

── Fold 1 ──
Fold 1 MAE: 7.0843
── Fold 2 ──
Fold 2 MAE: 7.2497
── Fold 3 ──
Fold 3 MAE: 7.0851
── Fold 4 ──
Fold 4 MAE: 7.1903
── Fold 5 ──
Fold 5 MAE: 7.1187

OOF MAE: 7.1456


In [9]:
test_id = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')['ID']

submission = pd.DataFrame({
    'ID'                        : test_id,
    'avg_delay_minutes_next_30m': test_preds
})

submission.to_csv('C:/Users/user/dacon/Smart_Logistics/results/submission_simple_layout_kfold.csv', index=False)
print(submission.head())

            ID  avg_delay_minutes_next_30m
0  TEST_000000                   15.832495
1  TEST_000001                   15.284694
2  TEST_000002                   18.999024
3  TEST_000003                   18.354112
4  TEST_000004                   17.349315
